In [ ]:
#%pip install -q --upgrade "boto3>=1.44.4,<2"


In [1]:
# Prints the wall-clock time under every cell.
%load_ext autotime

time: 169 µs (started: 2026-11-18 21:41:58 +05:30)


In [2]:
import csv
import io
import json
import math
import time
from datetime import datetime, timezone

import boto3
from botocore.config import Config
from botocore.exceptions import BotoCoreError, ClientError


time: 102 ms (started: 2026-11-18 21:41:58 +05:30)


In [3]:
REGION = "us-west-2"
ROLE_ARN = "arn:aws:iam::692859950400:role/service-role/AmazonSageMaker-ExecutionRole-20261118T211232"  # Paste your SageMaker execution ROLE ARN here.
PROJECT = "hotel-cancellation"
GROUP_NAME = f"{PROJECT}-model-group"
FIRST_VERSION = 1
NEXT_VERSION = 2  # Use 2 only after registering and reviewing version 2.
INSTANCE_TYPE = "ml.m5.large"
VARIANT_NAME = f"{PROJECT}-variant"

ERROR_RATE_LIMIT = 0.05     # 5% failed model invocations; not prediction error.
LATENCY_LIMIT_MS = 2000    # Classroom threshold; choose from measured latency in real use.
BAKE_SECONDS = 300         # Keep blue for 5 minutes after switching traffic.


time: 400 µs (started: 2026-11-18 21:51:34 +05:30)


In [5]:
session = boto3.Session(region_name=REGION)
sm = session.client("sagemaker")
cw = session.client("cloudwatch")
runtime = session.client("sagemaker-runtime", config=Config(read_timeout=70))
account_id = session.client("sts").get_caller_identity()["Account"]

print("Account:", account_id)
print("Region:", REGION)
assert ROLE_ARN.startswith(f"arn:aws:iam::{account_id}:role/"), "Set ROLE_ARN first."

Account: 692859950400
Region: us-west-2
time: 1.1 s (started: 2026-11-18 21:52:39 +05:30)


In [8]:
import uuid


run_id = uuid.uuid4().hex[:8]
blue_model_name = f"canvas-{run_id}-blue"
green_model_name = f"canvas-{run_id}-green"
blue_config_name = f"{blue_model_name}-config"
green_config_name = f"{green_model_name}-config"
package_prefix = f"arn:aws:sagemaker:{REGION}:{account_id}:model-package/{GROUP_NAME}"

endpoint_name = "hotel-cancellation-endpoint"


print("Endpoint:", endpoint_name)
print("Models:", blue_model_name, green_model_name)
print("Configurations:", blue_config_name, green_config_name)

Endpoint: hotel-cancellation-endpoint
Models: canvas-dd090e36-blue canvas-dd090e36-green
Configurations: canvas-dd090e36-blue-config canvas-dd090e36-green-config
time: 747 µs (started: 2026-11-18 21:53:43 +05:30)


In [10]:
for page in sm.get_paginator("list_model_packages").paginate(ModelPackageGroupName=GROUP_NAME):
    for version in page["ModelPackageSummaryList"]:
        print("Version:", version["ModelPackageVersion"],
              "| Status:", version["ModelPackageStatus"],
              "| Approval:", version["ModelApprovalStatus"])

Version: 1 | Status: Completed | Approval: PendingManualApproval
time: 892 ms (started: 2026-11-18 21:54:55 +05:30)


In [11]:
blue_package_arn = f"{package_prefix}/{FIRST_VERSION}"
blue_package = sm.describe_model_package(ModelPackageName=blue_package_arn)
assert blue_package["ModelPackageStatus"] == "Completed", "Wait for registration to finish."

print("Selected package:", blue_package_arn)
for position, container in enumerate(blue_package["InferenceSpecification"]["Containers"], 1):
    print(position, container["Image"])

Selected package: arn:aws:sagemaker:us-west-2:692859950400:model-package/hotel-cancellation-model-group/1
1 174368400705.dkr.ecr.us-west-2.amazonaws.com/sagemaker-data-wrangler-container:5.1.3
2 763104351884.dkr.ecr.us-west-2.amazonaws.com/autogluon-inference:1.1.0-cpu-py310-ubuntu20.04
time: 334 ms (started: 2026-11-18 21:55:12 +05:30)


In [12]:
sm.update_model_package(
    ModelPackageArn=blue_package_arn,
    ModelApprovalStatus="Approved",
    ApprovalDescription="Canvas validation results reviewed for the classroom deployment.",
)

{'ModelPackageArn': 'arn:aws:sagemaker:us-west-2:692859950400:model-package/hotel-cancellation-model-group/1',
 'ResponseMetadata': {'RequestId': '49654952-cf88-4236-891e-c3bf39684b26',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '49654952-cf88-4236-891e-c3bf39684b26',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '109',
   'date': 'Wed, 18 Nov 2026 16:29:33 GMT'},
  'RetryAttempts': 0}}

time: 1.04 s (started: 2026-11-18 21:59:32 +05:30)


In [13]:
sm.create_model(
    ModelName=blue_model_name,
    ExecutionRoleArn=ROLE_ARN,
    Containers=[{"ModelPackageName": blue_package_arn}],
)

{'ModelArn': 'arn:aws:sagemaker:us-west-2:692859950400:model/canvas-dd090e36-blue',
 'ResponseMetadata': {'RequestId': 'a5aab8c6-68ae-4ef8-95fd-1628397ac927',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'a5aab8c6-68ae-4ef8-95fd-1628397ac927',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '82',
   'date': 'Wed, 18 Nov 2026 16:30:11 GMT'},
  'RetryAttempts': 0}}

time: 1.55 s (started: 2026-11-18 22:00:10 +05:30)


In [15]:
variant_name = "main"


sm.create_endpoint_config(
    EndpointConfigName=blue_config_name,
    ProductionVariants=[{
        "VariantName": variant_name,
        "ModelName": blue_model_name,
        "InstanceType": INSTANCE_TYPE,
        "InitialInstanceCount": 1,
        "InitialVariantWeight": 1.0,
    }],
)

{'EndpointConfigArn': 'arn:aws:sagemaker:us-west-2:692859950400:endpoint-config/canvas-dd090e36-blue-config',
 'ResponseMetadata': {'RequestId': 'd4eb3626-5c8a-4f1f-a34e-f0d1f12fe3f7',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'd4eb3626-5c8a-4f1f-a34e-f0d1f12fe3f7',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '108',
   'date': 'Wed, 18 Nov 2026 16:31:13 GMT'},
  'RetryAttempts': 0}}

time: 1.04 s (started: 2026-11-18 22:01:12 +05:30)


In [ ]:
sm.describe_endpoint_config(EndpointConfigName=blue_config_name)
sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=blue_config_name)
sm.get_waiter("endpoint_in_service").wait(
    EndpointName=endpoint_name, WaiterConfig={"Delay": 20, "MaxAttempts": 180},
)

endpoint = sm.describe_endpoint(EndpointName=endpoint_name)
assert endpoint["EndpointConfigName"] == blue_config_name
print("First configuration is serving:", blue_config_name)